# DeBERTa on Triton with TensorRT FP16

Deploy [`cross-encoder/nli-deberta-v3-base`](https://huggingface.co/cross-encoder/nli-deberta-v3-base) to Amazon SageMaker using NVIDIA Triton Inference Server, accelerated by ONNX Runtime's TensorRT FP16 execution provider.

The notebook walks through:

1. Export the model to ONNX with baked-in postprocessing
2. Build a minimal Triton model repository (one static `config.pbtxt` with the TensorRT FP16 optimisation block)
3. Package and upload to S3
4. Deploy as a SageMaker **Inference Component** using the SageMaker Python SDK v3 `ModelBuilder`
5. Run a sample inference

Tested on a `conda_python3` kernel on a SageMaker notebook instance. Target endpoint instance is `ml.g5.2xlarge` (A10G).

## 1. Set up the environment

In [ ]:
!pip install -U pip "sagemaker>=3" boto3 transformers sentencepiece torch onnx onnxscript numpy

In [ ]:
import boto3
import json
import time
from pathlib import Path

from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = boto3.Session()
region = sess.region_name
role = get_execution_role()

print(f"Region: {region}")
print(f"Role:   {role}")

### Select the Triton container

See [AWS DLC available images](https://aws.github.io/deep-learning-containers/reference/available_images/) for the Triton image tag in your region.

In [ ]:
TRITON_ACCOUNT_ID = "763104351884"
base = "amazonaws.com.cn" if region.startswith("cn-") else "amazonaws.com"
triton_image_uri = (
    f"{TRITON_ACCOUNT_ID}.dkr.ecr.{region}.{base}"
    f"/sagemaker-tritonserver:24.09-py3"
)
print(triton_image_uri)

In [ ]:
ENDPOINT_NAME = "triton-deberta-tensorrt"
INSTANCE_TYPE = "ml.g5.2xlarge"

## 2. Export the model to ONNX

Run `workspace/export_model.py`, which:

- Loads `cross-encoder/nli-deberta-v3-base`
- Wraps it so the postprocessing (2-class softmax + entailment extraction) is part of the ONNX graph — no Python needed at inference time
- Exports to `workspace/nli_deberta/model.onnx`

In [ ]:
!python workspace/export_model.py

## 3. Build the Triton model repository

Triton expects a directory per model with a `config.pbtxt` and versioned weights under `{version}/`. We use the static [`workspace/config.pbtxt`](workspace/config.pbtxt) — its `optimization` block is where the ONNX graph level and TensorRT FP16 execution provider are declared.

```
triton-serve/
└── nli_deberta/
    ├── config.pbtxt
    └── 1/
        ├── model.onnx
        └── model.onnx.data   # external weights (if produced by ONNX export)
```

In [ ]:
import shutil

TRITON_REPO = Path("triton-serve")
if TRITON_REPO.exists():
    shutil.rmtree(TRITON_REPO)

version_dir = TRITON_REPO / "nli_deberta" / "1"
version_dir.mkdir(parents=True)

# Copy the static Triton config
shutil.copy("workspace/config.pbtxt", TRITON_REPO / "nli_deberta" / "config.pbtxt")

# Copy exported ONNX model (and external weights if present)
src = Path("workspace/nli_deberta/model.onnx")
shutil.copy(src, version_dir / "model.onnx")

external = src.with_suffix(src.suffix + ".data")
if external.exists():
    shutil.copy(external, version_dir / "model.onnx.data")

print(f"Triton repo at {TRITON_REPO.resolve()}")
for p in sorted(TRITON_REPO.rglob("*")):
    print(f"  {p.relative_to(TRITON_REPO)}")

### Inspect the config

The `optimization` block is what makes Triton worth using over a plain HuggingFace/PyTorch DLC. Everything else is standard Triton boilerplate.

In [ ]:
print(Path("workspace/config.pbtxt").read_text())

## 4. Package and upload to S3

In [ ]:
!tar -C triton-serve/ -czf model.tar.gz nli_deberta

sagemaker_session = Session(boto_session=sess)
bucket = sagemaker_session.default_bucket()
prefix = "triton-deberta-tensorrt"

sess.client("s3").upload_file("model.tar.gz", bucket, f"{prefix}/model.tar.gz")
model_uri = f"s3://{bucket}/{prefix}/model.tar.gz"

print(f"Uploaded to: {model_uri}")

## 5. Deploy as an Inference Component

Use SageMaker Python SDK v3's `ModelBuilder` with `ModelServer.TRITON`. `SAGEMAKER_TRITON_DEFAULT_MODEL_NAME` tells Triton which model to route requests to.

Inference Components let you allocate a subset of instance resources (GPU / CPU / memory) to this model, so multiple models can share an endpoint.

In [ ]:
from sagemaker.serve.model_builder import ModelBuilder, ModelServer
from sagemaker.core.inference_config import ResourceRequirements
from sagemaker.core.resources import Endpoint, InferenceComponent

MODEL_NAME = "nli_deberta"
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
sm_model_name = f"triton-deberta-{timestamp}"
ic_name = f"triton-deberta-ic-{timestamp}"

builder = ModelBuilder(
    image_uri=triton_image_uri,
    s3_model_data_url=model_uri,
    role_arn=role,
    env_vars={"SAGEMAKER_TRITON_DEFAULT_MODEL_NAME": MODEL_NAME},
    model_server=ModelServer.TRITON,
    instance_type=INSTANCE_TYPE,
    sagemaker_session=sagemaker_session,
)
builder.build(model_name=sm_model_name)

print(f"Deploying inference component '{ic_name}' ...")
builder.deploy(
    endpoint_name=ENDPOINT_NAME,
    inference_component_name=ic_name,
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    inference_config=ResourceRequirements(
        requests={"memory": 8192, "num_accelerators": 1, "num_cpus": 2, "copies": 1}
    ),
    wait=False,
)

Endpoint.get(ENDPOINT_NAME).wait_for_status("InService")
InferenceComponent.get(ic_name).wait_for_status("InService")

print(f"\nEndpoint:            {ENDPOINT_NAME}")
print(f"Inference component: {ic_name}")

## 6. Run a sample inference

Tokenize the input text against 50 candidate labels, send `[50, 128]` tensors to Triton, and read the entailment scores back. The label with the highest entailment probability is the predicted class.

In [ ]:
import numpy as np
from transformers import AutoTokenizer

MODEL_ID = "cross-encoder/nli-deberta-v3-base"
N_LABELS = 50
MAX_SEQ_LEN = 128
HYPOTHESIS_TEMPLATE = "This example is {}."

CANDIDATE_LABELS = [
    "positive review", "negative review", "neutral review",
    "product quality praise", "product quality complaint",
    "price complaint", "price praise", "recommendation",
    "warning to others", "shipping complaint", "shipping praise",
    "durability praise", "durability complaint", "customer service praise",
    "customer service complaint", "ease of use praise", "ease of use complaint",
    "design praise", "design complaint", "value for money",
    "overpriced", "underpriced", "feature request", "bug report",
    "question", "refund request", "return request", "warranty claim",
    "comparison with competitor", "first purchase", "repeat purchase",
    "gift purchase", "disappointment", "excitement", "frustration",
    "satisfaction", "loyalty expression", "switching intent",
    "technical issue", "packaging complaint", "packaging praise",
    "size complaint", "size praise", "color praise", "color complaint",
    "taste praise", "taste complaint", "comfort praise", "comfort complaint",
    "performance complaint",
]
assert len(CANDIDATE_LABELS) == N_LABELS

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
text = "This is a fantastic product! Highly recommend."

encoded = tokenizer(
    [text] * N_LABELS,
    [HYPOTHESIS_TEMPLATE.format(label) for label in CANDIDATE_LABELS],
    padding="max_length", truncation=True,
    max_length=MAX_SEQ_LEN, return_tensors="np",
)

payload = {
    "inputs": [
        {"name": "input_ids", "shape": [N_LABELS, MAX_SEQ_LEN],
         "datatype": "INT64", "data": encoded["input_ids"].tolist()},
        {"name": "attention_mask", "shape": [N_LABELS, MAX_SEQ_LEN],
         "datatype": "INT64", "data": encoded["attention_mask"].tolist()},
    ]
}

runtime = boto3.client("sagemaker-runtime", region_name=region)
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    InferenceComponentName=ic_name,
    ContentType="application/octet-stream",
    Body=json.dumps(payload),
)

scores = json.loads(response["Body"].read().decode("utf-8"))["outputs"][0]["data"]
top = int(np.argmax(scores))

print(f"Input: {text!r}")
print(f"Top label: {CANDIDATE_LABELS[top]!r} (score: {scores[top]:.3f})")

## 7. Clean up

Delete the inference component, endpoint, endpoint config, and model to stop charges.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model, InferenceComponent

InferenceComponent.get(ic_name).delete()
print(f"Deleted inference component: {ic_name}")
time.sleep(30)

Endpoint.get(ENDPOINT_NAME).delete()
print(f"Deleted endpoint: {ENDPOINT_NAME}")
time.sleep(30)

EndpointConfig.get(ENDPOINT_NAME).delete()
print(f"Deleted endpoint config: {ENDPOINT_NAME}")

Model.get(sm_model_name).delete()
print(f"Deleted model: {sm_model_name}")